# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic dataset metadata
metadata_json = dataset.metadata.to_json()
print(f"Dataset name: {getattr(dataset.metadata, 'name', None)}")
print(f"Description: {getattr(dataset.metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s in the dataset. This helps identify the structure and content that can be accessed.

**All entities are referenced by their unique `@id`.**

In [ ]:
# List available record sets, fields, and columns by their @id

def get_record_sets(ds):
    """Enumerate record sets (referenced by '@id') with their fields."""
    print("Available Record Sets:")
    for record_set in ds.metadata.record_sets:
        print(f"- Record Set: {record_set.name} (@id: {record_set.id})")
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - {column.name} (@id: {column.id}, source: {column.source})")
        print()

get_record_sets(dataset)

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. Use only the record set and field/column `@id`s from the overview above.

In [ ]:
# Choose record sets to extract (by @id)
# For this dataset, there is likely a main record set e.g. 'cr:RecordSet/mental_health_survey'. 
# We'll list all record sets and choose the first one as main for demonstration.

record_sets = [r.id for r in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets:
    print(f"Extracting records for Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Fields: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print("No records found in this record set.")# For demonstration, pick the first record set for further analysis.if len(record_sets) > 0 and record_sets[0] in dataframes:
    main_record_set_id = record_sets[0]
    print(f"Proceeding with main record set: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
else:
    print("No suitable record sets with records found. Please check the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps such as:
- Filtering records based on a numeric field
- Normalizing numeric values
- Grouping by a key field

**Example uses only `@id` for field/column names.**

In [ ]:
if 'main_df' in locals():
    # Try to choose a numeric field by its @id (e.g., GAD-7 score or similar)
    print("Available columns in DataFrame:")
    print(main_df.columns.tolist())
    
    # Example: let's assume GAD-7 total score field, replace with the actual @id for your data
    numeric_field_id = None
    for col in main_df.columns:
        # Try choosing a column that looks like a numeric field
        if 'score' in col.lower() or 'gad' in col.lower() or 'phq' in col.lower() or 'total' in col.lower():
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Fallback: just pick first numeric-looking field
        for col in main_df.columns:
            if pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_field_id = col
                break

    if numeric_field_id is not None:
        print(f"Using numeric field for analysis: {numeric_field_id}")

        # Filter records where the score > threshold
        threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print("No numeric field was found for analysis.")
    # Group by a key categorical field.
    group_field_id = None
    for col in main_df.columns:
        if ('gender' in col.lower() or 'sex' in col.lower()) and main_df[col].nunique() < 10:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found for grouping.")else:
    print("Main DataFrame not available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Example:** Numeric field distribution and group comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'main_df' in locals() and numeric_field_id is not None:
    # Show histogram of the numeric field
    plt.figure(figsize=(8,5))
    main_df[numeric_field_id].hist(bins=20)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If group field exists, show violin/box plot
    if group_field_id is not None:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
This notebook provided an overview of the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. 

- We loaded the dataset metadata and reviewed structure using record set and field `@id`s.
- Tabular data was extracted with dynamic referencing by `@id`.
- Performed basic EDA such as filtering, normalization, and grouping.
- Visualized key numeric variables by subgroups.

**Next steps:** This dataset can be further used for building mental health trend models, comparison across demographics, or other public health analyses. Always refer to data fields by their unique `@id` for reproducible and standards-based workflows.